# 📓 Audio AI Module 2: Dilated Causal Convolutions & Real Speech WaveNet (With Epoch Checkpoints)
Welcome to Module 2 of our Audio Models course! In this notebook, we build **WaveNet** (van den Oord et al., DeepMind, 2016) from scratch in PyTorch, train it on a **real human speech utterance for 1000 epochs**, and interactively compare how audio generation quality evolves across training checkpoints.

---

## 💡 Key Architectural Concepts

### 1. Causal Convolutions
An autoregressive model factorizes the joint probability of an audio waveform $x = \{x_1, x_2, \dots, x_T\}$ as:

$$p(x) = \prod_{t=1}^{T} p(x_t \mid x_1, \dots, x_{t-1})$$

To ensure $p(x_t)$ depends **only on past samples $x_{<t}$**, causal convolutions mask out all future time steps ($x_{>t}$).

### 2. Exponential Receptive Field via Dilated Convolutions
By applying **exponentially increasing dilation factors** $d = [1, 2, 4, 8, 16, 32, 64, 128, 256, 512]$ across stacked residual blocks, our network's receptive field grows exponentially to **$1,024$ samples (~$128\text{ ms}$ of acoustic history)**:

$$\text{Receptive Field} = 1 + \sum_{l=1}^{L} (K - 1) \cdot d_l$$

![Dilated Causal Convolution](https://raw.githubusercontent.com/basveeling/wavenet/master/wavenet.gif)
*(Diagram: Stacked 1D dilated causal convolutions growing exponentially across layers.)*

In [ ]:
import os
import copy
import urllib.request
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import torchaudio.transforms as T
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio, display, clear_output
import ipywidgets as widgets
from ipywidgets import interact, SelectionSlider

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Download & Load Real Speech Utterance
We load a $1.5\text{-second}$ real human speech audio clip, resampled to $8\text{ kHz}$ ($12,000$ continuous audio samples).

In [ ]:
# Load PyTorch official speech sample asset
audio_filename = "speech_sample.wav"
if not os.path.exists(audio_filename):
    # Hello
    url = "https://storage.googleapis.com/cloud-samples-data/speech/hello.wav"
    # Longer sentence
    # url = "https://cdn-media.huggingface.co/speech_samples/sample1.flac"
    urllib.request.urlretrieve(url, audio_filename)

    if url.endswith(".flac"):
        temp_waveform, temp_sr = torchaudio.load(url)
        torchaudio.save(audio_filename, temp_waveform, temp_sr)

waveform, sample_rate = torchaudio.load(audio_filename)

# Resample to 8 kHz for fast acoustic training
target_sr = 8000
if sample_rate != target_sr:
    waveform = T.Resample(orig_freq=sample_rate, new_freq=target_sr)(waveform)
    sample_rate = target_sr

# Ensure mono audio [1, T]
if waveform.shape[0] > 1:
    waveform = torch.mean(waveform, dim=0, keepdim=True)

# Trim to 1.5 seconds (12,000 samples)
max_samples = int(1.5 * sample_rate)
waveform = waveform[:, :max_samples]
waveform = waveform / torch.max(torch.abs(waveform)) # Normalize to [-1, 1]

speech_tensor = waveform.unsqueeze(0).to(device) # Shape: [1, 1, T]

print(f"Real Speech Utterance Loaded!")
print(f"Shape: {speech_tensor.shape}, Sample Rate: {sample_rate} Hz, Duration: {speech_tensor.shape[2]/sample_rate:.2f}s")
display(Audio(speech_tensor.squeeze().cpu().numpy(), rate=sample_rate))

## 2. Model Architecture Definitions
We implement `CausalDilatedConv1d`, `WaveNetResidualBlock`, and the complete `WaveNet` model.

In [ ]:
class CausalDilatedConv1d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=2, dilation=1):
        super(CausalDilatedConv1d, self).__init__()
        self.padding = (kernel_size - 1) * dilation
        self.conv = nn.Conv1d(
            in_channels,
            out_channels,
            kernel_size=kernel_size,
            dilation=dilation,
            padding=self.padding
        )
        
    def forward(self, x):
        out = self.conv(x)
        if self.padding != 0:
            out = out[:, :, :-self.padding]
        return out

class WaveNetResidualBlock(nn.Module):
    def __init__(self, residual_channels, gate_channels, kernel_size=2, dilation=1):
        super(WaveNetResidualBlock, self).__init__()
        self.filter_conv = CausalDilatedConv1d(residual_channels, gate_channels, kernel_size, dilation)
        self.gate_conv   = CausalDilatedConv1d(residual_channels, gate_channels, kernel_size, dilation)
        
        self.res_conv  = nn.Conv1d(gate_channels, residual_channels, kernel_size=1)
        self.skip_conv = nn.Conv1d(gate_channels, residual_channels, kernel_size=1)
        
    def forward(self, x):
        f = torch.tanh(self.filter_conv(x))
        g = torch.sigmoid(self.gate_conv(x))
        z = f * g  # Gated activation unit
        
        res  = self.res_conv(z)
        skip = self.skip_conv(z)
        
        return x + res, skip

class WaveNet(nn.Module):
    def __init__(self, in_channels=1, residual_channels=64, gate_channels=64, 
                 kernel_size=2, dilations=[1, 2, 4, 8, 16, 32, 64, 128, 256, 512]):
        super(WaveNet, self).__init__()
        self.kernel_size = kernel_size
        self.dilations = dilations
        self.receptive_field = 1 + sum((kernel_size - 1) * d for d in dilations)
        
        self.in_conv = CausalDilatedConv1d(in_channels, residual_channels, kernel_size=1, dilation=1)
        self.res_blocks = nn.ModuleList([
            WaveNetResidualBlock(residual_channels, gate_channels, kernel_size, d)
            for d in dilations
        ])
        self.out_head = nn.Sequential(
            nn.ReLU(),
            nn.Conv1d(residual_channels, residual_channels, kernel_size=1),
            nn.ReLU(),
            nn.Conv1d(residual_channels, in_channels, kernel_size=1),
            nn.Tanh()
        )
        
    def forward(self, x):
        x = self.in_conv(x)
        skip_sum = 0
        for block in self.res_blocks:
            x, skip = block(x)
            skip_sum = skip_sum + skip
        out = self.out_head(skip_sum)
        return out

wavenet_model = WaveNet().to(device)
print(f"WaveNet initialized! Receptive Field: {wavenet_model.receptive_field} samples.")

## 3. Single-Pass Training for 1000 Epochs with Checkpoint Storage
We train WaveNet for **1,000 total epochs**. During training, we automatically save state dict checkpoints into a dictionary `model_checkpoints[epoch]` at epochs `[100, 200, 300, 400, 500, 600, 700, 800, 900, 1000]`.

In [ ]:
optimizer = torch.optim.Adam(wavenet_model.parameters(), lr=1e-3)
total_epochs = 1000
checkpoint_epochs = [100, 200, 300, 400, 500, 600]
model_checkpoints = {}  # Dictionary storing model state dicts at checkpoint epochs

print(f"=== Training WaveNet for {total_epochs} epochs & saving checkpoints ===")
wavenet_model.train()

x_input = speech_tensor[:, :, :-1]
y_target = speech_tensor[:, :, 1:]

for epoch in range(1, total_epochs + 1):
    optimizer.zero_grad()
    y_pred = wavenet_model(x_input)
    
    loss = F.mse_loss(y_pred, y_target)
    loss.backward()
    optimizer.step()
    
    # Save checkpoint if current epoch is in checkpoint_epochs
    if epoch in checkpoint_epochs:
        model_checkpoints[epoch] = copy.deepcopy(wavenet_model.state_dict())
        print(f"Epoch {epoch:4d}/{total_epochs} | MSE Loss: {loss.item():.6f} -> Checkpoint Saved!")

print("Training Complete! Saved checkpoints for epochs:", list(model_checkpoints.keys()))

## 🎛️ 4. Interactive Checkpoint Visualizer & Audio Generation
Use the slider below to select a checkpoint epoch (e.g., $100, 200, 300, \dots, 1000$).
The widget dynamically loads the selected model weights, runs autoregressive generation, and displays the reconstructed waveform, Mel-Spectrogram, and playable audio so students can compare quality evolution over training time!

In [ ]:
# Evaluator model container
eval_model = WaveNet().to(device)
eval_model.eval()

def interactive_checkpoint_generation(selected_epoch=1000, num_gen_samples=4000):
    # 1. Load weights for selected epoch checkpoint
    eval_model.load_state_dict(model_checkpoints[selected_epoch])
    eval_model.eval()
    
    rf = eval_model.receptive_field
    generated_seq = speech_tensor[:, :, :rf].clone()
    
    # 2. Autoregressive sampling
    with torch.no_grad():
        for step in range(num_gen_samples):
            context = generated_seq[:, :, -rf:]
            next_pred = eval_model(context)[:, :, -1:]
            next_pred = torch.clamp(next_pred, -1.0, 1.0)
            generated_seq = torch.cat([generated_seq, next_pred], dim=2)
            
    gen_audio_np = generated_seq.squeeze().cpu().numpy()
    orig_audio_np = speech_tensor.squeeze().cpu().numpy()[:len(gen_audio_np)]
    
    clear_output(wait=True)
    print(f"Loaded Checkpoint: Epoch {selected_epoch}")
    print(f"Generated Duration: {len(gen_audio_np)/sample_rate:.2f} seconds ({len(gen_audio_np)} samples)")
    display(Audio(gen_audio_np, rate=sample_rate))
    
    # 3. Visualization
    fig, axes = plt.subplots(2, 1, figsize=(12, 6))
    
    # 1D Waveform plot
    axes[0].plot(orig_audio_np[:3000], color='#1f77b4', alpha=0.7, label="Ground Truth Real Speech")
    axes[0].plot(gen_audio_np[:3000], color='#ff7f0e', linestyle='--', alpha=0.8, label=f"WaveNet (Epoch {selected_epoch})")
    axes[0].axvline(x=rf, color='red', linestyle=':', label="Seed Window Boundary (1024 samples)")
    axes[0].set_title(f"Speech Waveform Comparison — Checkpoint Epoch {selected_epoch}")
    axes[0].set_ylabel("Amplitude")
    axes[0].legend()
    axes[0].grid(True, linestyle='--', alpha=0.5)
    
    # 2D Mel-Spectrogram plot
    mel_transform = T.MelSpectrogram(sample_rate=sample_rate, n_fft=512, hop_length=128, n_mels=64)
    gen_mel = torchaudio.transforms.AmplitudeToDB()(mel_transform(torch.tensor(gen_audio_np).unsqueeze(0)))
    
    im = axes[1].imshow(gen_mel.squeeze().numpy(), origin='lower', aspect='auto', cmap='magma')
    axes[1].set_title(f"Generated Mel-Spectrogram — Checkpoint Epoch {selected_epoch}")
    axes[1].set_xlabel("Time Frames")
    axes[1].set_ylabel("Mel Frequency Bins")
    fig.colorbar(im, ax=axes[1], format='%+2.0f dB')
    
    plt.tight_layout()
    plt.show()

# Selection slider widget for checkpoint epochs
interact(interactive_checkpoint_generation,
         selected_epoch=SelectionSlider(
             options=checkpoint_epochs,
             value=100,
             description='Epoch Checkpoint:'
         ));